# *Author: Victor Diallen*

## Importing Libraries

In [2]:
import requests
from bs4 import BeautifulSoup as soup
import pandas as pd
import time
import datetime
import csv
import smtplib
import ssl

## Parsing Page

In [19]:
headers = {'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_10_1) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/39.0.2171.95 Safari/537.36', "Accept-Encoding":"gzip, deflate", "Accept":"text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8", "DNT":"1","Connection":"close", "Upgrade-Insecure-Requests":"1"}
checkin_date = '2022-10-20'   # Put it in YYYY-MM-DD mode
checkout_date = '2022-12-20'   # Put it in YYYY-MM-DD mode

if checkin_date < checkout_date:
    url = 'https://www.booking.com/searchresults.html?label=gen173nr-1FCAEoggI46AdIM1gEaCCIAQGYATG4ARfIAQzYAQHoAQH4AQKIAgGoAgO4ApuP4JQGwAIB0gIkYWJiYzU5YjUtZmQyOC00NDg2LWE4NjItZGI4NjY2NWVkYzJi2AIF4AIB&sid=11549ce89bf44cc96ece2e065a80adc5&aid=304142&tmpl=searchresults&ac_click_type=b&ac_position=0&checkin='+checkin_date+'&checkout='+checkout_date+'&class_interval=1&dest_id=20088325&dest_type=city&dtdisc=0&group_adults=2&group_children=0&inac=0&index_postcard=0&label_click=undef&nflt=review_score%3D80&no_rooms=1&order=price&postcard=0&raw_dest_type=city&room1=A%2CA&sb_price_type=total&sb_travel_purpose=leisure&search_selected=1&shw_aparth=1&slp_r_match=0&srpvid=382d05bf6f0d00fa&ss=New+York%2C+New+York+State%2C+United+States&ss_all=0&ssb=empty&sshis=0&ssne=Manhattan&ssne_untouched=Manhattan&changed_currency=1&selected_currency=USD'
    page = requests.get(url, headers=headers)
    soup1 = soup(page.content, 'html.parser')
    soup2 = soup(soup1.prettify(), 'html.parser')

else:
    print("Change your dates. Checkout date must be after checking date.")

## Function to Check Price for Your Trip and Send Email if Price is Low

In [20]:
def tripToNYC():

    places = []
    reviews = []
    reviews_names = []
    prices = []
    dates = []
    
    ## Using BeautifulSoup to find elements inside page's content and appending them to lists

    container = soup2.find('div', {'id':'right'})

    place = container.findAll('div', {'data-testid':'title'})
    review = container.findAll('div', {'class':'b5cd09854e d10a6220b4'})
    review_name = container.findAll('div', {'class':'b5cd09854e f0d4d6a2f5 e46e88563a'})
    price = container.findAll('span', {'class':'fcab3ed991 bd73d13072'})

    for title in place:
        places.append(title.text.strip())

    for grade in review:
        reviews.append(grade.text.strip())

    for name in review_name:
        reviews_names.append(name.text.strip())

    for value in price:
        prices.append(value.text.strip().replace('US$',''))

    for today in range(0,((len(places)))):
        today = datetime.date.today()
        dates.append(today)

        
    ## Creating a DataFrame using Pandas and a CSV file 
    
    df = pd.DataFrame({
        'Accomodation':places,
        'Review Grade':reviews,
        'Review':reviews_names,
        'Price':prices,
        'Date':dates

    })

    df.to_csv('TripToNYC.csv', index=False)
    
    ## Creating a price alert and send email if price is below selected
    
    for chance in prices:
        if chance < '450':         ## Here you can select your price (this is in US dollars)

            port = 587  
            smtp_server = "smtp-mail.outlook.com"
            sender = "----------@hotmail.com"      ## Your email
            recipient = "_________@gmail.com"      ## Recipient's email
            sender_password = "xxxxxxxxx"          ## Your email's password

            message = """
            It's your chance to book a great accomodation in NYC!!
            """

            SSL_context = ssl.create_default_context()

            with smtplib.SMTP(smtp_server, port) as server:

                server.starttls(context=SSL_context)

                server.login(sender, sender_password)

                server.sendmail(sender, recipient, message)

                
## Timer to run script everyday

while True:
    tripToNYC()
    time.sleep(86400)

In [4]:
first = datetime.date(2022, 12, 25)

print(str(first))

2022-12-25


In [ ]:

while True:            
    tripToNYC()
    time.sleep(86400)